<a href="https://colab.research.google.com/github/atilimai/plant-ai-project/blob/main/notebooks/Ilgin_Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import pandas as pd
import os
from tqdm import tqdm

# Cihaz Ayarı
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

# Veri Yükleme
print("Orijinal veri seti Hugging Face'ten indiriliyor...")
hf_dataset = load_dataset("mohanty/PlantVillage", split="train")

print("Boran'ın CSV dosyaları okunuyor...")
train_df = pd.read_csv("data/splits/train_split.csv")
test_df = pd.read_csv("data/splits/test_split.csv")

# Performans Optimizasyonu (O(1) hızında arama yapmak için harita)
print("Veri indeksi oluşturuluyor (Bu işlem 1-2 dakika sürebilir)...")
path_to_idx = {item['image_path']: idx for idx, item in enumerate(hf_dataset)}

class PlantDataset(Dataset):
    def __init__(self, df, hf_dataset, path_to_idx, transform=None):
        self.df = df
        self.hf_dataset = hf_dataset
        self.path_to_idx = path_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['text']
        hf_idx = self.path_to_idx[img_path]
        sample = self.hf_dataset[hf_idx]

        image = sample['image'].convert("RGB")
        label = sample['label']

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = PlantDataset(train_df, hf_dataset, path_to_idx, transform)
test_dataset = PlantDataset(test_df, hf_dataset, path_to_idx, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Eğitim için {len(train_loader)} batch hazırlandı.")

In [ ]:
class PlantBaselineCNN(nn.Module):
    def __init__(self, num_classes=38):
        super(PlantBaselineCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

num_classes = 38
model = PlantBaselineCNN(num_classes=num_classes).to(device)

print("Model başarıyla oluşturuldu ve cihaza gönderildi.")
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3

print("Eğitim başlıyor...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

if not os.path.exists('models'):
    os.makedirs('models')

model_path = 'models/baseline_cnn_v1.pth'
torch.save(model.state_dict(), model_path)
print(f"\nModel başarıyla eğitildi ve ağırlıkları kaydedildi: {model_path}")